# Prompts

In [ ]:
import os
import sys
from pymongo import MongoClient

PROJECT_ROOT = "/home/ubuntu/projects/AI/git/dev/v03/law-document-sync-core-service"
sys.path.append(PROJECT_ROOT)
from constants import MongoDBCollectionConfig, MongoDBConfig, MigrateConfig, LLMsConfig
from core.common.llms import LLMs

llms = LLMs(LLMsConfig)


client = MongoClient(host=MongoDBConfig.HOST, 
                    port=MongoDBConfig.PORT,
                    username=MongoDBConfig.USERNAME,
                    password=MongoDBConfig.PASSWORD)

db = client[MigrateConfig.MIGRATE_CORE_DB]
law_documents_collection = db[MongoDBCollectionConfig.LAW_DOCUMENT_COLLECTION_NAME]
law_articles_collection = db[MongoDBCollectionConfig.LAW_ARTICLE_COLLECTION_NAME]


In [ ]:
import re
from typing import List, Dict, Any
from loguru import logger


def extract_regulate_entity(document_name: str, articles: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Tìm các điều luật có loại "phạm vi điều chỉnh" và "đối tượng áp dụng" từ danh sách các điều luật.
    
    Args:
        document_name (str): Tên văn bản pháp luật
        articles (List[Dict]): Danh sách các điều luật, mỗi điều luật có cấu trúc:
            {
                "article_number": str,  # Số điều luật
                "article_type": str,    # Loại điều luật
                "content": str,         # Nội dung điều luật
                "title": str            # Tiêu đề điều luật (nếu có)
            }
    
    Returns:
        Dict[str, Any]: Kết quả chứa:;
            - scope_articles: Danh sách điều luật phạm vi điều chỉnh
            - application_articles: Danh sách điều luật đối tượng áp dụng
            - prompt_input: Nội dung đã được format để truyền vào prompt
            - analysis_result: Kết quả phân tích (sẽ được điền sau khi gọi AI)
    """
    
    logger.info(f"Bắt đầu phân tích văn bản: {document_name}")
    logger.info(f"Số lượng điều luật đầu vào: {len(articles)}")
        
    # Các từ khóa để nhận diện phạm vi điều chỉnh
    scope_keywords = [
        "phạm vi điều chỉnh",
        "phạm vi áp dụng", 
        "điều chỉnh",
        "quy định về",
        "quy định này",
        "luật này"
    ]
    
    # Các từ khóa để nhận diện đối tượng áp dụng
    application_keywords = [
        "đối tượng áp dụng"        
    ]
    
    
    scope_article = ""    
    object_article = ""    
    for article in articles:
        article_title = article.get("article_title", "").lower()
        article_content = article.get("article_content", "").lower()
        content = f"{article_title}\n{article_content}"        
      
        # Kiểm tra phạm vi điều chỉnh
        is_scope = any(keyword in article_title for keyword in scope_keywords)
        if scope_article == "" and is_scope:
            scope_article = content


        # Kiểm tra đối tượng áp dụng  
        is_application = any(keyword in article_title for keyword in application_keywords)        
        if object_article == "" and is_application:
            object_article = content


    logger.debug(f"scope_article: {scope_article}")
    logger.debug(f"object_article: {object_article}")


    # Bước 2: Tạo prompt input
    prompt = f"""
**Bạn là một trợ lý AI chuyên phân tích văn bản quy phạm pháp luật (QPPL) của Việt Nam.** Nhiệm vụ của bạn là xác định **đối tượng điều chỉnh** của văn bản pháp luật dựa trên các đầu vào được cung cấp.

## **Định nghĩa**
- **Đối tượng điều chỉnh**: Các quan hệ xã hội chung mà văn bản pháp luật tác động đến để điều chỉnh hành vi của các chủ thể, được mô tả tổng quát theo lĩnh vực hoặc phạm vi của văn bản. Ví dụ: “các quan hệ xã hội liên quan đến quản lý, sử dụng, và giao dịch đất đai” hoặc “các quan hệ xã hội liên quan đến quản lý lý lịch tư pháp”.
- **Đối tượng áp dụng**: Các chủ thể chịu tác động của văn bản (cá nhân, tổ chức, cơ quan nhà nước), không phải là đối tượng điều chỉnh, nhưng giúp xác định các bên tham gia trong các quan hệ xã hội.

## **Đầu vào**
- **Tên văn bản**: {document_name}
- **Điều luật về phạm vi điều chỉnh**: {scope_article}
- **Điều luật về đối tượng áp dụng**: {object_article}
  
## **Hướng dẫn**
1. **Phân tích tên văn bản**:
   - Xác định lĩnh vực pháp luật tổng quát (ví dụ: đất đai, lý lịch tư pháp, doanh nghiệp) dựa trên tên văn bản.
   - Ví dụ: “Luật Đất đai” → Liên quan đến các quan hệ xã hội về đất đai.
2. **Phân tích điều luật về phạm vi điều chỉnh**:
   - Xác định các quan hệ xã hội chung mà văn bản tác động đến, dựa trên các từ khóa như “quản lý,” “giao dịch,” “quyền và nghĩa vụ.”
   - Mô tả tổng quát, tránh chi tiết hóa thành các loại quan hệ cụ thể (như hành chính, dân sự).
3. **Phân tích điều luật về đối tượng áp dụng**:
   - Xác định tất cả các chủ thể (cá nhân, tổ chức, cơ quan nhà nước) tham gia vào các quan hệ xã hội.
   - Gộp các chủ thể trùng lặp, thống nhất loại (ví dụ: cơ quan nhà nước, cá nhân) và vai trò (ví dụ: quản lý, sử dụng).
4. **Tổng hợp đối tượng điều chỉnh**:
   - Kết hợp thông tin từ tên văn bản, phạm vi điều chỉnh, và đối tượng áp dụng để mô tả các quan hệ xã hội chung theo lĩnh vực của văn bản.
   - Đảm bảo mô tả tổng quát, đúng với nội dung văn bản và không phân chia thành các loại quan hệ cụ thể.
5. **Định dạng kết quả**:
   - Trả về kết quả dưới dạng JSON, chỉ chứa thông tin về **đối tượng điều chỉnh** (các quan hệ xã hội chung) và **ngành luật**.
   - Cấu trúc JSON:
    {{
       "doi_tuong_dieu_chinh": "[Mô tả tổng quát các quan hệ xã hội chung theo lĩnh vực của văn bản]",
       "nganh_luat": "[Tên ngành luật, ví dụ: đất đai, hành chính - tư pháp, kinh tế]"
     }}

## **Lưu ý**
- **Chỉ sử dụng thông tin** có trong tên văn bản và nội dung hai điều luật được cung cấp – không suy luận từ nguồn bên ngoài.
- **Liệt kê tất cả chủ thể** xuất hiện trong điều luật về đối tượng áp dụng, đảm bảo không bỏ sót.
- **Không trùng lặp chủ thể**: Nếu cùng tên (ví dụ: “cơ quan nhà nước” xuất hiện nhiều lần), gộp lại và thống nhất loại (ví dụ: cơ quan nhà nước) và vai trò (ví dụ: quản lý đất đai).
- **Không thêm giải thích**: Kết quả chỉ chứa dữ liệu JSON, không bao gồm mô tả hoặc giải thích ngoài định dạng yêu cầu.
- **Ngôn ngữ pháp lý**: Sử dụng thuật ngữ pháp lý tiếng Việt chính xác (ví dụ: “quan hệ xã hội liên quan đến quản lý đất đai”).
- **Mô tả tổng quát**: Đảm bảo đối tượng điều chỉnh được mô tả ở mức chung, đúng với lĩnh vực của văn bản, không chi tiết hóa thành các loại quan hệ cụ thể.


## **Ví dụ**
**Đầu vào**:
- **Tên văn bản**: Luật Đất đai 2024
- **Điều luật về phạm vi điều chỉnh**: Luật này quy định về chế độ sở hữu đất đai, quyền hạn và trách nhiệm của Nhà nước đại diện chủ sở hữu toàn dân về đất đai; quản lý, sử dụng đất đai; quyền và nghĩa vụ của tổ chức, cá nhân, hộ gia đình trong việc sử dụng đất đai và thực hiện các giao dịch liên quan đến đất đai.
- **Điều luật về đối tượng áp dụng**: 
  Luật này áp dụng đối với cơ quan nhà nước thực hiện chức năng quản lý nhà nước về đất đai; tổ chức, hộ gia đình, cá nhân sử dụng đất; cộng đồng dân cư; tổ chức tôn giáo; người Việt Nam định cư ở nước ngoài; tổ chức, cá nhân nước ngoài có liên quan đến việc sử dụng đất đai tại Việt Nam.

**Kết quả**:
{{

  "doi_tuong_dieu_chinh": "Các quan hệ xã hội liên quan đến quản lý, sử dụng và giao dịch đất đai",
  "nganh_luat": "Đất đai"
}}

## **Yêu cầu thực hiện**
- Phân tích đầu vào và trả về kết quả dưới dạng JSON theo định dạng đã nêu.
- Đảm bảo kết quả ngắn gọn, chính xác, và phù hợp với pháp luật Việt Nam.
- **Bắt đầu phân tích ngay.**"""

    response = llms.llms(prompt)
    try:
        result = llms.llms_post_process(response)
    except Exception:
        result = response
    return result 


# Testing

In [13]:
def get_regulate_entity(doc_id):
    document = law_documents_collection.find_one({'doc_id': doc_id})
    document_name = document.get("doc_title", "")
    articles = list(law_articles_collection.find({"doc_id": doc_id}))

    return extract_regulate_entity(document_name, articles)

In [14]:
doc_id = "296884"

get_regulate_entity(doc_id)

2025-10-17 17:08:13.686 | INFO     | __main__:extract_regulate_entity:28 - Bắt đầu phân tích văn bản: Bộ luật tố tụng hình sự 2015
2025-10-17 17:08:13.687 | INFO     | __main__:extract_regulate_entity:29 - Số lượng điều luật đầu vào: 501
2025-10-17 17:08:13.693 | DEBUG    | __main__:extract_regulate_entity:66 - scope_article: điều 1. phạm vi điều chỉnh
bộ luật tố tụng hình sự quy định trình tự, thủ tục tiếp nhận, giải quyết nguồn tin về tội phạm, khởi tố, điều tra, truy tố, xét xử và một số thủ tục thi hành án hình sự; nhiệm vụ, quyền hạn và mối quan hệ giữa các cơ quan có thẩm quyền tiến hành tố tụng; nhiệm vụ, quyền hạn và trách nhiệm của người có thẩm quyền tiến hành tố tụng; quyền và nghĩa vụ của người tham gia tố tụng, cơ quan, tổ chức, cá nhân; hợp tác quốc tế trong tố tụng hình sự.
2025-10-17 17:08:13.693 | DEBUG    | __main__:extract_regulate_entity:67 - object_article: 
2025-10-17 17:08:13.694 | DEBUG    | core.common.llms:llms:41 - LLMs USE OLLAMA
2025-10-17 17:08:15.467 | DE

dict_string: {
  "doi_tuong_dieu_chinh": "Các quan hệ xã hội liên quan đến trình tự, thủ tục tiếp nhận, giải quyết nguồn tin về tội phạm, khởi tố, điều tra, truy tố, xét xử và thi hành án hình sự, cũng như nhiệm vụ, quyền hạn, trách nhiệm của người có thẩm quyền tiến hành tố tụng và quyền, nghĩa vụ của người tham gia tố tụng, cơ quan, tổ chức, cá nhân trong hoạt động tố tụng hình sự và hợp tác quốc tế trong tố tụng hình sự",
  "nganh_luat": "Hành chính - tư pháp"
}


{'doi_tuong_dieu_chinh': 'Các quan hệ xã hội liên quan đến trình tự, thủ tục tiếp nhận, giải quyết nguồn tin về tội phạm, khởi tố, điều tra, truy tố, xét xử và thi hành án hình sự, cũng như nhiệm vụ, quyền hạn, trách nhiệm của người có thẩm quyền tiến hành tố tụng và quyền, nghĩa vụ của người tham gia tố tụng, cơ quan, tổ chức, cá nhân trong hoạt động tố tụng hình sự và hợp tác quốc tế trong tố tụng hình sự',
 'nganh_luat': 'Hành chính - tư pháp'}